# URJA Telemetry Synth — 10M rows (Scale Test)
Generates 5 years × 50 assets at 15min, tests TimescaleDB `time_bucket` like `daily_rollup` worker.


In [ ]:
!test -d URJA || git clone https://github.com/ravikumarve/URJA.git --depth 1
%cd URJA
!pip -q install pandas numpy --progress-bar off 2>&1 | tail -1
import pandas as pd, numpy as np, time
print("ready")

In [ ]:
N_ASSETS=50; YEARS=5; PER_DAY=96  # 15min
N = N_ASSETS * YEARS * 365 * PER_DAY
print(f"Generating {N:,} rows (~{N*32/1e6:.0f} MB)...")
t0=time.time()
df=pd.DataFrame({
 "ts": pd.date_range("2021-01-01", periods=YEARS*365*PER_DAY, freq="15min").repeat(N_ASSETS),
 "asset_id": np.tile([f"a-{i}" for i in range(1,N_ASSETS+1)], YEARS*365*PER_DAY),
 "generation_kw": np.random.normal(3200,500, N).clip(0,5000),
})
print(f"Done {time.time()-t0:.1f}s", df.memory_usage(deep=True).sum()/1e6, "MB")
df.head(2)

In [ ]:
# Benchmark time_bucket-like rollup (pandas groupby)
t0=time.time()
daily=df.groupby([pd.Grouper(key="ts", freq="1D"), "asset_id"])["generation_kw"].mean()
print(f"Daily rollup: {len(daily):,} groups in {time.time()-t0:.2f}s")
print(daily.head(3))
print("On Latitude TimescaleDB this is `time_bucket('1 day', ts)` — should be <2s for 10M")

In [ ]:
df.sample(100000).to_csv("/tmp/telemetry_100k.csv", index=False)
from google.colab import files; files.download("/tmp/telemetry_100k.csv")
print("↓ 100K sample for seed.py testing")